<a href="https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

**Chosen lane:** Refresh / Content Opportunity Scoring  
**Notebook purpose:** Frame the problem and inspect the safe starter dataset; **not** a new trained model.

[Open the repository notebook in Colab](https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Core first, AI second:** These paragraphs are a suggested starting point. Review, personalize and make sure you can explain the decisions yourself before committing. The starter CSV contains anonymized content only; never add client-identifying information or new datasets to git.

## 1. My lane as an ML task (type)

I chose **Refresh / Content Opportunity Scoring**. The primary ML task is **ranking/scoring**: recommend which existing content pages a human editor should review first, given limited review time. A binary *declining-content* classification proxy can help evaluate the ranking, but the **decision** is which pages to prioritize, not simply whether a page has declined.

**Decision:** Which visible content pages should receive an editorial review first?  
**User and action:** A content editor or SEO analyst reviews high-ranked pages for outdated information, search-intent fit, or opportunities to improve their content. A ranking suggests a queue; it does **not** automatically rewrite or publish content.  
**Cost of mistakes:** False positives waste editor time; false negatives may delay review of genuinely declining pages.

I will start with pages receiving **at least 100 search impressions over 90 days** so the ranking emphasizes a measurable visibility opportunity. This is a practical minimum-volume choice, not a claim that 100 is the universally correct cutoff.

In [6]:
# Section 1 — Load the small, public starter data from MY GitHub copy.
# No private dataset, Hugging Face token, or pip installation needed.
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

def locate_repo():
    # Supports running locally from the repo, and in a fresh Colab runtime.
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / DATA_REL).is_file():
            return p
    if "google.colab" in sys.modules:
        destination = Path("/content/flyrank-ml-internship")
        if not (destination / DATA_REL).is_file():
            if destination.exists():
                raise FileNotFoundError("Clone directory exists without starter CSV; check /content/flyrank-ml-internship")
            subprocess.run([
                "git", "clone", "--depth", "1",
                "https://github.com/Jhonas05/flyrank-ml-internship.git",
                str(destination)
            ], check=True)
        return destination
    raise FileNotFoundError("Please run from the starter repository, or use Google Colab.")

repo_root = locate_repo()
df = pd.read_csv(repo_root / DATA_REL)
lane = df.loc[
    (df["impressions_90d"] >= 100) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset=["content_id"]).copy()

assert len(lane) >= 50, "Need at least 50 pages for Precision@50."
print(f"Starter dataset: {len(df):,} rows x {df.shape[1]} columns")
print(f"Eligible review pages (>=100 impressions, >=90 days old): {len(lane):,}")
print("Chosen task: ranking/scoring pages for human editorial review")

Starter dataset: 30,000 rows x 44 columns
Eligible review pages (>=100 impressions, >=90 days old): 22,006
Chosen task: ranking/scoring pages for human editorial review


## 2. Target or proxy

The **initial proxy** is `declining_proxy = 1` for pages whose `trend_direction` is `"down"`, and `0` otherwise. In the starter dataset, `trend_direction` is a **rule-defined category derived from observed changes in impressions** between the recent 30 days and the preceding 30 days. This is **not** an independently measured future outcome, so I will not describe the starter framing as accurate prediction of future decline.

This proxy lets me demonstrate the ML loop now: data → score → ranked queue → check how many high-ranked pages show current decline → human review. In a stronger later experiment, I would use **earlier-window features** to rank pages against an **outcome measured in a subsequent 30-day window** using the warehouse data, if the historical coverage supports it.

**Leakage guard:** `trend_direction`, `trend_pct`, and values used to construct the label must not be model input features for predicting that label. `content_id` and `client_id` are for grouping/splitting only. Later, I must also check that *all* model features were available **before** the future-outcome period.

In [7]:
# Section 2 — Build and inspect the explicitly limited current-window proxy.
lane["declining_proxy"] = (
    lane["trend_direction"].astype(str).str.lower().eq("down")
).astype("int8")
summary = lane["declining_proxy"].value_counts().reindex([0, 1], fill_value=0)
print(f"Not declining by this proxy: {summary.loc[0]:,}")
print(f"Declining by this proxy:     {summary.loc[1]:,}")
print(f"Observed proxy prevalence:  {lane['declining_proxy'].mean():.1%}")
assert set(lane["declining_proxy"].unique()).issubset({0, 1})

Not declining by this proxy: 8,854
Declining by this proxy:     13,152
Observed proxy prevalence:  59.8%


## 3. Success metric

I choose **Precision@50 (P@50)**. It answers a practical question: **Of the first 50 pages suggested for editor review, how many meet our declining-content proxy?** A score of `0.70`, for example, would mean 35 of the top 50 meet the proxy; it would **not** mean those edits are proven to cause improvements.

**Proposed success criterion for a later fair comparison:** The ML-driven ranking should beat a pre-defined, transparent fixed-rule baseline by **at least 0.05 absolute P@50** (5 percentage points) on a **client-holdout evaluation**. This threshold is a project goal, not a result already achieved. We should also report the proxy prevalence and inspect errors so we do not confuse a strong metric with a useful action.

Below I score a **simple illustrative rule** on the current starter slice. This is **descriptive practice on the same snapshot**, *not* a client-holdout score or proof of predictive generalization.

In [8]:
# Section 3 — Define P@50 and compute a transparent illustrative baseline.
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    assert len(scores) == len(labels) and len(scores) >= k
    top_indices = np.argsort(-scores, kind="stable")[:k]
    return float(labels[top_indices].mean())

# Fixed rule: prioritize pages not updated for 180+ days,
# ranked by their 90-day search impressions.
lane["fixed_rule_score"] = (
    (lane["days_since_last_update"] >= 180).astype(int)
    * lane["impressions_90d"].fillna(0)
)
illustrative_p50 = precision_at_k(
    lane["fixed_rule_score"], lane["declining_proxy"], 50
)
print(f"Illustrative current-snapshot fixed rule P@50: {illustrative_p50:.3f}")
print(f"Proxy prevalence in this lane slice:         {lane['declining_proxy'].mean():.3f}")
print("Target for later client-holdout comparison: ML P@50 >= baseline P@50 + 0.050")
print("Reminder: today's value is descriptive, not out-of-sample performance.")

Illustrative current-snapshot fixed rule P@50: 0.720
Proxy prevalence in this lane slice:         0.598
Target for later client-holdout comparison: ML P@50 >= baseline P@50 + 0.050
Reminder: today's value is descriptive, not out-of-sample performance.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page** summarized over the starter dataset's trailing 90-day window. I use pages with at least 100 impressions and at least 90 days of content age. I check that each `content_id` occurs only once, but do **not** publish IDs or client names in the displayed table.

The table below shows the unit of analysis and some observable signals. `ctr` is stored in **percentage-point units** (e.g., `0.76` means **0.76%**, not 76%). `avg_position = 0` means *no position data*, not a real top ranking.

In [9]:
# Section 4 — Show actual rows without exposing even pseudonymous IDs.
from IPython.display import display

assert lane["content_id"].is_unique, "Expected one row per content item."
columns_to_show = [
    "content_type", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update",
    "content_age_days", "declining_proxy"
]
print(f"One row = one content page. Distinct pages: {lane['content_id'].nunique():,}")
print(f"Displayed preview: {min(8, len(lane))} anonymized rows")
display(lane[columns_to_show].head(8).reset_index(drop=True))

One row = one content page. Distinct pages: 22,006
Displayed preview: 8 anonymized rows


,content_type,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,content_age_days,declining_proxy
0,keyword article,3803,29,0.76,10.6,20,187,1
1,keyword article,15320,7,0.05,20.3,25,445,1
2,keyword article,12581,11,0.09,36.5,20,141,1
3,keyword article,11751,58,0.49,6.2,22,463,0
4,keyword article,19140,24,0.13,44.0,14,263,1
5,keyword article,3970,1,0.03,8.5,20,147,1
6,keyword article,1724,1,0.06,21.2,22,445,0
7,keyword article,32574,29,0.09,46.0,20,90,1


## 5. Why ML beats a fixed rule here

A fixed rule such as "old + visible = review" is understandable and makes a useful **baseline**, but it cannot easily capture interactions among freshness, visibility, search position, click-through rate, content type, and engagement. Some old pages perform well, while some recently updated pages may still be declining. A learned ranking **might** capture patterns that simple thresholds miss, but it earns its complexity only if it outperforms the fixed rule on held-out clients and produces actionable, reviewable reasons.

**Editor-facing decision:** A high-ranked page becomes a *review recommendation*, not an automatic edit. The editor checks whether the page needs factual updates, a better match to search intent, clearer headings, or other relevant improvements. False positives cost review time; false negatives may miss time-sensitive content opportunities. No causal claim about Google rankings or traffic gains is justified by these observations.

The simple counts below illustrate how many pages a fixed rule would flag. They do **not** claim that an ML model has already beaten the rule.

In [10]:
# Section 5 — Illustrate the editor action from a transparent rule.
lane["review_flag_rule"] = (
    (lane["days_since_last_update"] >= 180) &
    (lane["impressions_90d"] >= 500)
)
flagged = int(lane["review_flag_rule"].sum())
print(f"Pages meeting the illustrative 'old and visible' rule: {flagged:,}")
print(f"Share of eligible pages: {flagged / len(lane):.1%}")
print("Suggested action: human editor reviews flagged pages for freshness and relevance.")
print("Next ML step (later assignment): compare ranked P@50 on a client-holdout split.")

Pages meeting the illustrative 'old and visible' rule: 17
Share of eligible pages: 0.1%
Suggested action: human editor reviews flagged pages for freshness and relevance.
Next ML step (later assignment): compare ranked P@50 on a client-holdout split.


## 6. Self-check

**Before submitting, I will verify and tick these items myself:**

- [ ] I can explain why my task is **ranking/scoring**, and which editor decision it improves.
- [ ] I distinguish the **rule-defined current-window proxy** from an observed **future** outcome.
- [ ] I chose **Precision@50** in advance, stated a provisional success threshold, and did not call same-snapshot results validation.
- [ ] I can show the **real dataframe** and explain that one row represents one content page.
- [ ] I can explain the cost of wrong recommendations and why ML must beat a transparent rule before adoption.
- [ ] I checked that the notebook ran top to bottom with **no errors**, without adding raw data or secrets to GitHub.
- [ ] I personally reviewed/reworded the framing, saved executed outputs under **`work/notebooks/w02_ml_task_framing.ipynb`**, and verified them on GitHub.

**One-paragraph ML frame (personalize before submission):** For content editors deciding which existing pages to review first, I propose a ranked content opportunity queue from pseudonymized search and freshness signals, initially assessed using a declining-content *proxy* and **Precision@50**. Wrong recommendations waste editorial effort or delay needed reviews. ML is worth testing because multiple page signals interact in ways a single threshold cannot express; however, any improvement must be validated on held-out clients, with time-aligned features for genuinely future outcomes. Results will be framed as observed and decision-support, not causal proof.